# When auth retries look periodic

A synthetic demonstration of a false-positive trap in auth timing analysis.

> **Finding:** a classical periodicity test can report strong cadence in nonperiodic auth retries when the events arrive in short clusters. The Fourier transform is doing its job; the test's white-spectrum null model is not describing the data-generating process.

## Data disclosure

This notebook contains only deterministic synthetic timestamps. It contains no log records, network identifiers, accounts, hosts, addresses, or values derived from an operational environment.

## Question and scope

Repeated authentication attempts naturally form short clusters: one initiating event followed by several retries a few seconds apart. Those clusters are not themselves a repeating schedule. But they introduce dependence between nearby event times, which colours the background spectrum.

This notebook asks a narrow question: **does Fisher's g-test retain its nominal false-positive rate when the null process contains retries?** It compares two synthetic nulls with the same target event volume and observation geometry:

- a homogeneous Poisson process, whose binned counts have a flat expected spectrum;
- a stationary parent-child process, where every independent parent produces two to five retries delayed by one to ten seconds.

Every timestamp is generated from the public seed below. This is a method demonstration, not an auth detector and not an attack classifier.

Fisher introduced the largest-periodogram-ordinate test in [Tests of Significance in Harmonic Analysis](https://doi.org/10.1098/rspa.1929.0151). A modern treatment of the same clustered-background failure mode, in another event domain, is Park, Kiraly, and Bourne's [Periodic seismicity detection without declustering](https://arxiv.org/abs/2101.11533).

In [ ]:
# ── Cell 0 ── reproducible configuration & imports
from __future__ import annotations

import hashlib
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import gammaln, logsumexp
from scipy.stats import beta

PUBLIC_SEED = 20260803
BIN_SECONDS = 5.0
REFERENCE_INTERVAL_SECONDS = 60.0
MIN_EVENTS = 32
MIN_PERIOD_SECONDS = 15.0
MIN_OBSERVED_CYCLES = 8.0
ALPHA_FWER = 0.01
EDGES_PER_INVOCATION = 20
INVOCATIONS = 200

plt.style.use("seaborn-v0_8-whitegrid")

## Two nonperiodic event generators

The `reference_support` and `REFERENCE_INTERVAL_SECONDS` values determine the target event count and observation span; they do **not** put a cadence into either null. Each invocation uses independently derived random streams, so loop order cannot change the population.

The clustered process includes a ten-second burn-in. Parents are generated before the visible window so that retries near the left boundary have the same opportunity to appear as retries elsewhere. A series is redrawn only when it falls below the common 32-event eligibility floor.

In [ ]:
# ── Cell 1 ── homogeneous and clustered synthetic nulls
def _canonical_json(value: object) -> bytes:
    return (
        json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
        + "\n"
    ).encode("utf-8")


def rng_for(*parts: object) -> np.random.Generator:
    payload = _canonical_json([PUBLIC_SEED, *parts])
    seed = int.from_bytes(hashlib.blake2b(payload, digest_size=16).digest(), "big")
    return np.random.Generator(np.random.PCG64(seed))


def observation_span(reference_support: int) -> float:
    return 1.25 * (reference_support - 1) * REFERENCE_INTERVAL_SECONDS


def homogeneous_poisson_times(
    reference_support: int,
    window_seconds: float,
    rng: np.random.Generator,
) -> np.ndarray:
    while True:
        count = int(rng.poisson(reference_support))
        if count >= MIN_EVENTS:
            return np.sort(rng.uniform(0.0, window_seconds, count))


def clustered_retry_times(
    reference_support: int,
    window_seconds: float,
    rng: np.random.Generator,
) -> np.ndarray:
    mean_cluster_size = 4.5  # one parent plus a discrete-uniform 2..5 retries
    parent_rate = reference_support / (mean_cluster_size * window_seconds)
    burn_in_seconds = 10.0

    while True:
        parent_count = int(rng.poisson(parent_rate * (window_seconds + burn_in_seconds)))
        parents = rng.uniform(-burn_in_seconds, window_seconds, parent_count)
        events: list[float] = []
        for parent in parents:
            events.append(float(parent))
            retry_count = int(rng.integers(2, 6))
            events.extend(parent + rng.uniform(1.0, 10.0, retry_count))

        visible = np.sort(np.asarray(events, dtype=np.float64))
        visible = visible[(visible >= 0.0) & (visible < window_seconds)]
        if visible.size >= MIN_EVENTS:
            return visible


GENERATORS = {
    "homogeneous Poisson": homogeneous_poisson_times,
    "clustered retries": clustered_retry_times,
}

In [ ]:
# ── Cell 2 ── look at the event geometry before testing it
def binned_counts(times: np.ndarray, window_seconds: float) -> np.ndarray:
    bin_count = int(math.ceil(window_seconds / BIN_SECONDS))
    counts = np.zeros(bin_count, dtype=np.float64)
    indices = np.floor(times / BIN_SECONDS).astype(np.int64)
    np.add.at(counts, indices, 1.0)
    return counts


example_support = 128
example_window = observation_span(example_support)
examples = {
    name: generator(example_support, example_window, rng_for("example", name))
    for name, generator in GENERATORS.items()
}

view_seconds = 30 * 60
fig, axes = plt.subplots(2, 2, figsize=(13, 5.5), sharex="col")
for row, (name, times) in enumerate(examples.items()):
    visible = times[times < view_seconds]
    axes[row, 0].eventplot(visible / 60.0, lineoffsets=0.5, linelengths=0.8)
    axes[row, 0].set_yticks([])
    axes[row, 0].set_ylabel(name)

    counts = binned_counts(times, example_window)[: int(view_seconds / BIN_SECONDS)]
    minutes = np.arange(counts.size) * BIN_SECONDS / 60.0
    axes[row, 1].step(minutes, counts, where="post")
    axes[row, 1].set_ylabel("events / 5 s")

axes[0, 0].set_title("Event times: first 30 minutes")
axes[0, 1].set_title("The same events after five-second binning")
axes[-1, 0].set_xlabel("minutes")
axes[-1, 1].set_xlabel("minutes")
fig.suptitle("Both nulls are nonperiodic; only one contains retries", y=1.02)
fig.tight_layout()
plt.show()

## Fisher's g-test

For non-DC periodogram ordinates $I_1, \ldots, I_m$, Fisher's statistic is

$$g = \frac{\max_k I_k}{\sum_k I_k}.$$

Under the classical white-spectrum null, the exact finite tail is

$$P(G \ge g) = \sum_{j=1}^{\lfloor 1/g \rfloor} (-1)^{j-1} {m \choose j}(1-jg)^{m-1}.$$

The implementation below evaluates that alternating sum in log space. Each simulated invocation contains twenty independently generated identity edges. Their raw p-values receive one Holm step-down correction at family-wise $\alpha=0.01$. A rejection only surfaces when the dominant period is at least 15 seconds and the active event span contains at least eight such periods.

In [ ]:
# ── Cell 3 ── Fisher finite-tail test + run-wide Holm correction
def fisher_tail_probability(m: int, g: float) -> float:
    if m < 1 or not math.isfinite(g):
        raise ValueError("m must be positive and g must be finite")
    if g <= 1.0 / m:
        return 1.0
    if g >= 1.0:
        return 0.0

    upper = min(m, int(math.floor(1.0 / g)))
    indices = np.arange(1, upper + 1, dtype=np.float64)
    bases = 1.0 - indices * g
    keep = bases > 0.0
    indices = indices[keep]
    bases = bases[keep]
    logs = (
        gammaln(m + 1.0)
        - gammaln(indices + 1.0)
        - gammaln(m - indices + 1.0)
        + (m - 1.0) * np.log(bases)
    )
    signs = np.where(indices.astype(np.int64) % 2 == 1, 1.0, -1.0)
    log_absolute, sign = logsumexp(logs, b=signs, return_sign=True)
    value = float(sign * np.exp(log_absolute))
    if sign > 0.0 and math.isfinite(value):
        return min(1.0, max(0.0, value))

    # Near-uniform spectra can lose their sign through catastrophic cancellation.
    # They are far from the rejection region, so one is the conservative value.
    return 1.0


def score_times(times: np.ndarray, window_seconds: float) -> dict[str, float | bool]:
    counts = binned_counts(times, window_seconds)
    centred = counts - counts.mean()
    ordinates = np.square(np.abs(np.fft.rfft(centred)[1:]))
    energy = float(ordinates.sum())
    if not energy > 0.0:
        return {"p_value": 1.0, "passes_product_gates": False}

    dominant_index = int(np.argmax(ordinates)) + 1
    g = float(ordinates[dominant_index - 1] / energy)
    period_seconds = counts.size * BIN_SECONDS / dominant_index
    observed_cycles = float((times[-1] - times[0]) / period_seconds)
    return {
        "p_value": fisher_tail_probability(ordinates.size, g),
        "g": g,
        "period_seconds": period_seconds,
        "observed_cycles": observed_cycles,
        "passes_product_gates": (
            period_seconds >= MIN_PERIOD_SECONDS
            and observed_cycles >= MIN_OBSERVED_CYCLES
        ),
    }


def holm_rejections(p_values: list[float], alpha: float = ALPHA_FWER) -> np.ndarray:
    order = sorted(range(len(p_values)), key=lambda index: (p_values[index], index))
    rejected = np.zeros(len(p_values), dtype=bool)
    rejection_open = True
    for rank, index in enumerate(order):
        passes = p_values[index] <= alpha / (len(order) - rank)
        rejected[index] = rejection_open and passes
        rejection_open = rejection_open and passes
    return rejected


assert math.isclose(fisher_tail_probability(16, 0.10), 0.9998972230, rel_tol=1e-9)
assert math.isclose(fisher_tail_probability(16, 0.20), 0.5071289911, rel_tol=1e-9)
assert math.isclose(fisher_tail_probability(128, 0.05), 0.1775295231, rel_tol=1e-9)

## The null spectrum is not white

Averaging many spectra makes the assumption failure visible. Each periodogram is divided by its own mean ordinate before averaging. Under the homogeneous null, the curve stays near one. Retry clustering shifts expected energy toward lower frequencies, so every frequency is no longer exchangeable under a single white-noise tail.

In [ ]:
# ── Cell 4 ── show where the white-spectrum assumption breaks
def mean_normalized_spectrum(
    name: str,
    reference_support: int = 128,
    replicates: int = 100,
) -> tuple[np.ndarray, np.ndarray]:
    window_seconds = observation_span(reference_support)
    rows = []
    for replicate in range(replicates):
        times = GENERATORS[name](
            reference_support,
            window_seconds,
            rng_for("spectrum", name, reference_support, replicate),
        )
        counts = binned_counts(times, window_seconds)
        ordinates = np.square(np.abs(np.fft.rfft(counts - counts.mean())[1:]))
        rows.append(ordinates / ordinates.mean())
    frequencies = np.fft.rfftfreq(counts.size, d=BIN_SECONDS)[1:]
    return frequencies, np.mean(rows, axis=0)


fig, ax = plt.subplots(figsize=(10.5, 4.2))
for name in GENERATORS:
    frequencies, spectrum = mean_normalized_spectrum(name)
    cycles_per_hour = frequencies * 3600.0
    ax.plot(cycles_per_hour, spectrum, label=name, linewidth=1.5)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="white expectation")
ax.set_xlim(0, 360)
ax.set_ylim(bottom=0)
ax.set_xlabel("frequency (cycles/hour)")
ax.set_ylabel("mean power / series mean power")
ax.set_title("Retry clustering colours the background spectrum")
ax.legend()
fig.tight_layout()
plt.show()

## False-positive experiment

The scoring unit below is an invocation containing twenty eligible edges. An invocation is a false positive when **any** edge survives both Holm correction and the period/span gates. This family-wise unit matters: a tool runs across many identities at once, not against one hand-picked series.

The three reference supports expose a useful interaction. As more events make the coloured spectrum easier to resolve, a miscalibrated test can become more confidently wrong.

In [ ]:
# ── Cell 5 ── family-wise false positives over 20-edge invocations
def clopper_pearson_interval(false_invocations: int, invocations: int) -> tuple[float, float]:
    lower = (
        0.0
        if false_invocations == 0
        else float(beta.ppf(0.025, false_invocations, invocations - false_invocations + 1))
    )
    upper = (
        1.0
        if false_invocations == invocations
        else float(beta.ppf(0.975, false_invocations + 1, invocations - false_invocations))
    )
    return lower, upper


def run_false_positive_family(name: str, reference_support: int) -> dict[str, object]:
    window_seconds = observation_span(reference_support)
    false_invocations = 0
    for invocation in range(INVOCATIONS):
        scores = []
        for edge in range(EDGES_PER_INVOCATION):
            times = GENERATORS[name](
                reference_support,
                window_seconds,
                rng_for("monte-carlo", name, reference_support, invocation, edge),
            )
            scores.append(score_times(times, window_seconds))

        rejected = holm_rejections([float(score["p_value"]) for score in scores])
        surfaced = any(
            rejected[index] and bool(score["passes_product_gates"])
            for index, score in enumerate(scores)
        )
        false_invocations += int(surfaced)

    lower, upper = clopper_pearson_interval(false_invocations, INVOCATIONS)
    return {
        "null process": name,
        "reference support": reference_support,
        "false invocations": false_invocations,
        "invocations": INVOCATIONS,
        "family-wise false-positive rate": false_invocations / INVOCATIONS,
        "95% exact lower": lower,
        "95% exact upper": upper,
    }


rows = [
    run_false_positive_family(name, support)
    for support in (32, 128, 1024)
    for name in GENERATORS
]
results = pd.DataFrame(rows)
display_results = results.copy()
for column in (
    "family-wise false-positive rate",
    "95% exact lower",
    "95% exact upper",
):
    display_results[column] = display_results[column].map(lambda value: f"{value:.1%}")
display_results

In [ ]:
# ── Cell 6 ── calibration result
supports = [32, 128, 1024]
x = np.arange(len(supports), dtype=np.float64)
width = 0.34
fig, ax = plt.subplots(figsize=(10.5, 4.8))
for offset, name in enumerate(GENERATORS):
    subset = results[results["null process"] == name].set_index("reference support").loc[supports]
    rates = subset["family-wise false-positive rate"].to_numpy()
    lower = subset["95% exact lower"].to_numpy()
    upper = subset["95% exact upper"].to_numpy()
    positions = x + (offset - 0.5) * width
    bars = ax.bar(positions, rates, width, label=name, yerr=[rates - lower, upper - rates], capsize=4)
    for bar, rate in zip(bars, rates, strict=True):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            min(rate + 0.035, 1.03),
            f"{rate:.1%}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
ax.axhline(ALPHA_FWER, color="black", linestyle="--", linewidth=1, label="nominal 1%")
ax.set_xticks(x, supports)
ax.set_ylim(0, 1.10)
ax.set_xlabel("reference support")
ax.set_ylabel("family-wise false-positive rate")
ax.set_title(f"Fisher g-test calibration over {INVOCATIONS} synthetic 20-edge invocations")
ax.yaxis.set_major_formatter(lambda value, position: f"{value:.0%}")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[-1:] + handles[:-1], labels[-1:] + labels[:-1], loc="upper left")
fig.tight_layout()
plt.show()

## Interpretation

The homogeneous control stays near the nominal family-wise rate. The retry-cluster null does not. Its false-positive rate rises with support even though no generator contains a repeating schedule. More data makes the violated assumption easier to see; it does not make the p-value more valid.

A larger synthetic gate that motivated this demonstration used 500 invocations, twenty edges per invocation, and mixed support/window profiles. Fisher's g-test surfaced at least one false periodicity in **484 of 500 clustered-retry invocations (96.8%)**. The smaller experiment above is intentionally standalone: it uses a simple stationary generator so the mechanism is inspectable in one notebook.

This result does **not** show that periodic auth behaviour is uninteresting. It shows that the significance claim must be calibrated against a background process capable of producing ordinary retries. Plausible ways forward include a cluster-aware spectral background, a preregistered empirical null with an explicit cost budget, or a narrow heuristic whose assumptions are visible in its output.

It also does not validate a replacement detector. A paired-gap heuristic explored alongside this test produced five false invocations in 500 on one clustered family and zero in 500 on six other null families, but its held-out positive arm was not measured. That establishes restraint on those nulls, not detection power or operational value.

The durable lesson is smaller than a detector design: **a named statistical method is only as trustworthy as the null model attached to its p-value.**